In [1]:
import os
import subprocess
import time
import shutil
import csv
from datetime import datetime
import stat

import re

import ast
import json
import shutil
import builtins

In [2]:
# KONFIGURASI

input_file = r"D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\asli\nim_github.txt"
projects_folder = r"D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\code_mhs"
os.makedirs(projects_folder, exist_ok=True)

In [3]:
#LOG Folder

log_folder = r"D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\log"
os.makedirs(log_folder, exist_ok=True)

In [4]:
# LOG CLONE
log_file = os.path.join(log_folder, "log_clone.csv")

# SETUP LOG

if not os.path.exists(log_file):
    with open(log_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["timestamp", "nim", "url", "status", "message"])

def write_log(nim, url, status, message):
    with open(log_file, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            nim,
            url,
            status,
            message
        ])

# Pengumpulan Data

In [5]:
# BACA DATASET
# dataset berupa nim | url
def load_dataset(file_path):

    dataset = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split("|")

            if len(parts) != 2:
                continue

            nim = parts[0].strip()
            url = parts[1].strip()

            dataset.append((nim, url))

    return dataset

In [6]:
# HANYA SIMPAN .py & .ipynb

def keep_only_code_files(repo_path):

    for root, dirs, files in os.walk(repo_path):

        # Jangan masuk folder .git
        if ".git" in dirs:
            dirs.remove(".git")

        for file in files:
            if not (file.endswith(".py") or file.endswith(".ipynb")):
                try:
                    os.remove(os.path.join(root, file))
                except:
                    pass

    # Hapus folder kosong
    for root, dirs, files in os.walk(repo_path, topdown=False):
        if not os.listdir(root):
            try:
                os.rmdir(root)
            except:
                pass


In [7]:
# Remove folder gagal

def remove_readonly(func, path, exc_info):
    """
    Menghapus atribut read-only lalu retry delete.
    """
    try:
        os.chmod(path, stat.S_IWRITE)
        func(path)
    except Exception as e:
        print(f"Gagal paksa hapus: {path}")
        print(f"->   Alasan: {str(e)}")

In [8]:
# CLONE FUNCTION

def clone_repo(nim, url):

    global success_count, fail_count, skip_count

    # Jika NIM kosong
    if not nim:
        print(f"[SKIP] URL tanpa NIM → {url}")
        write_log("", url, "SKIPPED", "NIM kosong")
        skip_count += 1
        return

    # Bersihkan URL jika ada /tree/
    if "/tree/" in url:
        url = url.split("/tree/")[0]

    target_path = os.path.join(projects_folder, nim)

    # Jika sudah pernah clone
    if os.path.exists(target_path):
        print(f"[SKIP] {nim} sudah ada.")
        write_log(nim, url, "SKIPPED", "Folder sudah ada")
        skip_count += 1
        return

    print(f"[CLONE] {url}")

    try:
        subprocess.run(
            ["git", "clone", url, target_path],
            check=True,
            timeout=300,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.PIPE
        )

        # Simpan hanya .py dan .ipynb
        keep_only_code_files(target_path)

        print(f"   ✅ Berhasil clone {nim}")
        write_log(nim, url, "SUCCESS", "Clone berhasil")
        success_count += 1

        time.sleep(1)

    except subprocess.CalledProcessError as e:

        error_msg = e.stderr.decode(errors="ignore")

        print(f"   ❌ Gagal clone {nim}")
        print(f"   Alasan:\n{error_msg}")

        write_log(nim, url, "FAILED", error_msg)

        # Hapus folder jika setengah clone
        if os.path.exists(target_path):
            try:
                shutil.rmtree(target_path, onerror=remove_readonly)
                print("   🧹 Folder clone dihapus")
            except Exception as delete_error:
                print("   ⚠ Gagal hapus folder clone")
                print(str(delete_error))

        fail_count += 1

    except subprocess.TimeoutExpired:

        print(f"   ⏰ Timeout clone {nim}")
        write_log(nim, url, "FAILED", "Timeout")

        if os.path.exists(target_path):
            try:
                shutil.rmtree(target_path)
            except:
                pass

        fail_count += 1

In [9]:
dataset = load_dataset(input_file)

total_url = len(dataset)
success_count = 0
fail_count = 0
skip_count = 0

print(f"\nTotal URL dalam dataset: {total_url}\n")

for nim, url in dataset:
    clone_repo(nim, url)

print("\n===== RINGKASAN CLONING =====")
print(f"Total URL      : {total_url}")
print(f"Berhasil clone : {success_count}")
print(f"Gagal clone    : {fail_count}")
print(f"Skipped        : {skip_count}")
print("================================")
print(f"Log tersimpan di: {log_file}")


Total URL dalam dataset: 57

[SKIP] 2341720040 sudah ada.
[SKIP] 2341720131 sudah ada.
[SKIP] 2341720070 sudah ada.
[SKIP] 2341720153 sudah ada.
[SKIP] 2241720092 sudah ada.
[SKIP] 2341720187 sudah ada.
[SKIP] 2341720144 sudah ada.
[SKIP] 2341720041 sudah ada.
[SKIP] 2341720111 sudah ada.
[SKIP] 2341720218 sudah ada.
[CLONE] https://github.com/fajrulsantoso/244107023010_ML_2025
   ❌ Gagal clone 244107023010
   Alasan:
Cloning into 'D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\code_mhs\244107023010'...
error: invalid path 'JS08 /JS08'
fatal: unable to checkout working tree
You can inspect what was checked out with 'git status'
and retry with 'git restore --source=HEAD :/'


   🧹 Folder clone dihapus
[SKIP] 2341720081 sudah ada.
[SKIP] 2341720176 sudah ada.
[SKIP] 2341720003 sudah ada.
[SKIP] 2341720109 sudah ada.
[SKIP] 2341720035 sudah ada.
[SKIP] 2341720096 sudah ada.
[SKIP] 2341720057 sudah ada.
[SKIP] 2341720168 sudah ada.
[SKIP] 2341720088 sudah ada.
[SKIP] 2341720009 sudah ada.
[SKIP

## normalisasi struktur direktori

In [ ]:
normalized_folder = projects_folder + "_normalized"
os.makedirs(normalized_folder, exist_ok=True)

In [10]:
def remove_empty_folders(base_path):
    """
    Menghapus semua folder kosong kecuali folder utama mahasiswa.
    """
    for root, dirs, files in os.walk(base_path, topdown=False):

        # Jangan hapus root utama mahasiswa
        if root == base_path:
            continue

        if not os.listdir(root):
            try:
                os.rmdir(root)
            except:
                pass

In [11]:
# log untuk normalisasi struktur

log_file = os.path.join(log_folder, "log_normalisasiStrukturNamaFile.csv")

# SETUP LOG

if not os.path.exists(log_file):
    with open(log_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["timestamp", "nim", "action", "detail"])

def write_log(nim, action, detail):
    with open(log_file, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            nim,
            action,
            detail
        ])

In [ ]:
def identify_module_from_name(name):

    name_upper = name.upper()

    # PRIORITAS KHUSUS DULU
    if "KUIS" in name_upper:
        return "kuis"

    if "UTS" in name_upper:
        return "uts"

    if "PBL" in name_upper:
        return "pbl"

    match_kel = re.search(r"KELOMPOK\s*0?(\d+)?", name_upper)
    if match_kel:
        number = match_kel.group(1)
        number = int(number) if number else 1
        return f"kelompok{number:02d}"
    
    match_tp = re.search(r"TP\s*0?(\d+)?", name_upper)
    if match_tp:
        number = match_tp.group(1)
        number = int(number) if number else 1
        return f"tp{number:02d}"

    # BARU JSxx
    match_js = re.search(r"JS\s?0?(\d+)", name_upper)
    if match_js:
        return f"m{int(match_js.group(1)):02d}"

    return None

In [ ]:
# NORMALISASI FILE

def normalize_file_name(file_name, module_folder, counter_dict):

    name_upper = file_name.upper()
    ext = os.path.splitext(file_name)[1].lower()

    # PRAKTIKUM Pxx
    match_p = re.search(r"\bP\s?0?(\d+)", name_upper)
    if match_p:
        number = int(match_p.group(1))
        return f"p{number:02d}{ext}"

    # TPxx
    match_tp = re.search(r"TP\s*0?(\d+)?", name_upper)
    if match_tp:
        number = match_tp.group(1)
        number = int(number) if number else 1
        return f"tp{number:02d}{ext}"

    # kelompok
    match_kel = re.search(r"KELOMPOK\s*0?(\d+)?", name_upper)
    if match_kel:
        number = match_kel.group(1)
        number = int(number) if number else 1
        return f"kelompok{number:02d}{ext}"
    
    # KUIS / UTS
    if module_folder in ["kuis", "uts", "pbl"]:
        return f"{module_folder}{ext}"

    return file_name

In [ ]:
# NORMALISASI REPO

def normalize_repository(nim_path):

    nim = os.path.basename(nim_path)
    print(f"\n🔎 Normalisasi {nim}")

    counter_dict = {}
    unclassified_path = os.path.join(nim_path, "unclassified")
    os.makedirs(unclassified_path, exist_ok=True)

    # =========================
    # STEP 1 - Rename folder level atas
    # =========================
    for folder in os.listdir(nim_path):

        old_folder_path = os.path.join(nim_path, folder)

        if not os.path.isdir(old_folder_path):
            continue

        new_module = identify_module_from_name(folder)

        if new_module:
            new_folder_path = os.path.join(nim_path, new_module)

            if old_folder_path != new_folder_path:
                # Jika folder tujuan belum ada → rename biasa
                if not os.path.exists(new_folder_path):
                    os.rename(old_folder_path, new_folder_path)
                    write_log(nim, "RENAME_FOLDER", f"{folder} → {new_module}")

                else:
                    # Jika sudah ada → merge isi folder
                    for item in os.listdir(old_folder_path):
                        src_item = os.path.join(old_folder_path, item)
                        dst_item = os.path.join(new_folder_path, item)

                        if not os.path.exists(dst_item):
                            shutil.move(src_item, dst_item)
                        else:
                            base, ext = os.path.splitext(item)
                            counter = 1
                            while True:
                                new_name = f"{base}_{counter}{ext}"
                                new_dst = os.path.join(new_folder_path, new_name)
                                if not os.path.exists(new_dst):
                                    shutil.move(src_item, new_dst)
                                    break
                                counter += 1

                    # Hapus folder lama setelah merge
                    if os.path.exists(old_folder_path):
                        shutil.rmtree(old_folder_path, ignore_errors=True)

                    write_log(nim, "MERGE_FOLDER", f"{folder} → {new_module}")

    # =========================
    # STEP 2 - Pindahkan & rename file
    # =========================
    for root, dirs, files in os.walk(nim_path):

        for file in files:

            if not (file.endswith(".py") or file.endswith(".ipynb")):
                continue

            file_path = os.path.join(root, file)

            # Tentukan modul dari folder atau file
            module = identify_module_from_name(root)
            if not module:
                module = identify_module_from_name(file)

            if not module:
                target_folder = unclassified_path
            else:
                target_folder = os.path.join(nim_path, module)
                os.makedirs(target_folder, exist_ok=True)

            if module not in counter_dict:
                counter_dict[module] = 0

            new_name = normalize_file_name(file, module, counter_dict)
            target_path = os.path.join(target_folder, new_name)

            if file_path != target_path:
                shutil.move(file_path, target_path)
                write_log(nim, "MOVE_RENAME_FILE", f"{file} → {module}/{new_name}")

    # =========================
    # STEP 3 - Hapus folder kosong
    # =========================
    remove_empty_folders(nim_path)

    print(f"   ✅ Selesai {nim}")

In [15]:
# MAIN LOOP
print("\n🚀 Mulai normalisasi struktur & nama file...")

for student in os.listdir(projects_folder):

    student_path = os.path.join(projects_folder, student)

    if os.path.isdir(student_path):
        normalize_repository(student_path)

print("\n===== NORMALISASI SELESAI =====")
print(f"Log tersimpan di: {log_file}")


🚀 Mulai normalisasi struktur & nama file...

🔎 Normalisasi 2241720092
   ✅ Selesai 2241720092

🔎 Normalisasi 2341720003
   ✅ Selesai 2341720003

🔎 Normalisasi 2341720005
   ✅ Selesai 2341720005

🔎 Normalisasi 2341720007
   ✅ Selesai 2341720007

🔎 Normalisasi 2341720009
   ✅ Selesai 2341720009

🔎 Normalisasi 2341720011
   ✅ Selesai 2341720011

🔎 Normalisasi 2341720017
   ✅ Selesai 2341720017

🔎 Normalisasi 2341720028
   ✅ Selesai 2341720028

🔎 Normalisasi 2341720032
   ✅ Selesai 2341720032

🔎 Normalisasi 2341720035
   ✅ Selesai 2341720035

🔎 Normalisasi 2341720040
   ✅ Selesai 2341720040

🔎 Normalisasi 2341720041
   ✅ Selesai 2341720041

🔎 Normalisasi 2341720047
   ✅ Selesai 2341720047

🔎 Normalisasi 2341720057
   ✅ Selesai 2341720057

🔎 Normalisasi 2341720062
   ✅ Selesai 2341720062

🔎 Normalisasi 2341720070
   ✅ Selesai 2341720070

🔎 Normalisasi 2341720072
   ✅ Selesai 2341720072

🔎 Normalisasi 2341720081
   ✅ Selesai 2341720081

🔎 Normalisasi 2341720083
   ✅ Selesai 2341720083

🔎 No

## preprocessing code

In [16]:
# folder dan log

preprocessed_folder = projects_folder + "_preprocessed"
os.makedirs(preprocessed_folder, exist_ok=True)

log_file = os.path.join(log_folder, "log_preprocessing.csv")

# SETUP LOG

if not os.path.exists(log_file):
    with open(log_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["timestamp", "file_path", "status", "message"])

def write_log(file_path, status, message):
    with open(log_file, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            file_path,
            status,
            message
        ])

In [17]:
# EKSTRAK NOTEBOOK

def extract_code_from_notebook(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            notebook = json.load(f)
    except:
        return None

    code_blocks = []

    for cell in notebook.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        source = "".join(cell.get("source", []))
        code_blocks.append(source)

    return "\n".join(code_blocks)

In [18]:
# VALIDASI CODE

def validate_code(code):
    if not code or not code.strip():
        return False, "File kosong"

    try:
        ast.parse(code)
        return True, "Valid"
    except SyntaxError as e:
        return False, f"SyntaxError baris {e.lineno}: {e.msg}"
    except Exception as e:
        return False, str(e)

In [19]:
# CLEANING CODE

def basic_cleaning(code):
    """
    Cleaning awal sebelum validasi:
    - Hapus magic command
    - Hapus karakter aneh
    """
    code = re.sub(r"^\s*%.*$", "", code, flags=re.MULTILINE)
    code = re.sub(r"^\s*!.*$", "", code, flags=re.MULTILINE)
    return code

def advanced_cleaning(code):
    """
    Cleaning lanjutan setelah validasi
    - Hapus docstring
    - Hapus komentar
    - Hapus baris kosong berlebih
    """
    try:
        tree = ast.parse(code)
    except:
        return None

    # Hapus docstring
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.ClassDef, ast.Module)):
            if (
                node.body
                and isinstance(node.body[0], ast.Expr)
                and isinstance(node.body[0].value, ast.Str)
            ):
                node.body.pop(0)

    cleaned = ast.unparse(tree)

    # Hapus komentar
    cleaned = re.sub(r"#.*", "", cleaned)

    # Hapus baris kosong
    lines = cleaned.splitlines()
    lines = [line.rstrip() for line in lines if line.strip() != ""]

    return "\n".join(lines)

In [20]:
# NORMALISASI IDENTIFIER

class IdentifierNormalizer(ast.NodeTransformer):

    def __init__(self):
        self.var_map = {}
        self.func_map = {}
        self.arg_map = {}

        self.var_counter = 1
        self.func_counter = 1
        self.arg_counter = 1

        self.builtin_names = set(dir(builtins))

    def visit_FunctionDef(self, node):

        if node.name not in self.func_map:
            self.func_map[node.name] = f"func{self.func_counter}"
            self.func_counter += 1

        node.name = self.func_map[node.name]

        for arg in node.args.args:
            if arg.arg not in self.arg_map:
                self.arg_map[arg.arg] = f"arg{self.arg_counter}"
                self.arg_counter += 1
            arg.arg = self.arg_map[arg.arg]

        self.generic_visit(node)
        return node

    def visit_Name(self, node):

        if node.id in self.builtin_names:
            return node

        if node.id in self.arg_map:
            node.id = self.arg_map[node.id]
            return node

        if node.id not in self.var_map:
            self.var_map[node.id] = f"var{self.var_counter}"
            self.var_counter += 1

        node.id = self.var_map[node.id]
        return node

def normalize_identifiers(code):
    try:
        tree = ast.parse(code)
        normalizer = IdentifierNormalizer()
        tree = normalizer.visit(tree)
        ast.fix_missing_locations(tree)
        return ast.unparse(tree)
    except Exception as e:
        return None



In [21]:
# Loop PREPROCESSING

print("\n🚀 Mulai preprocessing...")

valid_count = 0
invalid_count = 0
empty_count = 0

for root, dirs, files in os.walk(projects_folder):

    relative_path = os.path.relpath(root, projects_folder)
    target_root = os.path.join(preprocessed_folder, relative_path)
    os.makedirs(target_root, exist_ok=True)

    for file in files:

        if not (file.endswith(".py") or file.endswith(".ipynb")):
            continue

        source_path = os.path.join(root, file)

        # === Ambil kode ===
        if file.endswith(".py"):
            with open(source_path, "r", encoding="utf-8") as f:
                code = f.read()
        else:
            code = extract_code_from_notebook(source_path)

        if code is None:
            write_log(source_path, "FAILED", "Tidak bisa membaca file")
            continue

        # === CLEANING AWAL (magic command dulu) ===
        code = basic_cleaning(code)

        # === VALIDASI ===
        is_valid, reason = validate_code(code)
        if not is_valid:
            write_log(source_path, "INVALID", reason)
            if "kosong" in reason.lower():
                empty_count += 1
            else:
                invalid_count += 1
            continue

        # === CLEANING LANJUTAN ===
        cleaned = advanced_cleaning(code)
        if not cleaned:
            write_log(source_path, "FAILED", "Cleaning gagal")
            continue

        # === NORMALISASI IDENTIFIER ===
        normalized = normalize_identifiers(cleaned)
        if not normalized:
            write_log(source_path, "FAILED", "Identifier normalization gagal")
            continue

        # === SIMPAN ===
        new_filename = file.replace(".ipynb", ".py")
        target_path = os.path.join(target_root, new_filename)

        with open(target_path, "w", encoding="utf-8") as f:
            f.write(normalized)

        write_log(source_path, "SUCCESS", "Preprocessing berhasil")
        valid_count += 1

print("\n===== RINGKASAN PREPROCESSING =====")
print(f"File valid & diproses : {valid_count}")
print(f"File kosong           : {empty_count}")
print(f"File syntax error     : {invalid_count}")
print("====================================")
print(f"Hasil tersimpan di: {preprocessed_folder}")


🚀 Mulai preprocessing...

===== RINGKASAN PREPROCESSING =====
File valid & diproses : 1323
File kosong           : 41
File syntax error     : 27
Hasil tersimpan di: D:\PUTRI\D4\SEMESTER 8 (4C)\skripsi\code\code_mhs_preprocessed


In [22]:
def count_code_files(folder):
    total_py = 0
    total_ipynb = 0

    for root, dirs, files in os.walk(folder):
        for file in files:
            if file.endswith(".py"):
                total_py += 1
            elif file.endswith(".ipynb"):
                total_ipynb += 1

    return total_py, total_ipynb


before_py, before_ipynb = count_code_files(projects_folder)

print("=== SEBELUM PREPROCESSING ===")
print("Jumlah .py     :", before_py)
print("Jumlah .ipynb  :", before_ipynb)
print("Total file kode:", before_py + before_ipynb)

preprocessed_folder = projects_folder + "_preprocessed"

after_py, after_ipynb = count_code_files(preprocessed_folder)

print("\n=== SETELAH PREPROCESSING ===")
print("Jumlah .py     :", after_py)
print("Jumlah .ipynb  :", after_ipynb)
print("Total file kode:", after_py + after_ipynb)

total_before = before_py + before_ipynb
total_after = after_py + after_ipynb

print("\n=== PERBANDINGAN ===")
print("Total sebelum :", total_before)
print("Total sesudah :", total_after)
print("Total terbuang:", total_before - total_after)

if total_before > 0:
    persen_hilang = ((total_before - total_after) / total_before) * 100
    print("Persentase terbuang: {:.2f}%".format(persen_hilang))

=== SEBELUM PREPROCESSING ===
Jumlah .py     : 401
Jumlah .ipynb  : 1134
Total file kode: 1535

=== SETELAH PREPROCESSING ===
Jumlah .py     : 3271
Jumlah .ipynb  : 0
Total file kode: 3271

=== PERBANDINGAN ===
Total sebelum : 1535
Total sesudah : 3271
Total terbuang: -1736
Persentase terbuang: -113.09%
